# Nemotron-3 Diarization — who spoke when on a free T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/Nemotron-3-Diarization.ipynb)

[`nvidia/Nemotron-3-Diarization`](https://huggingface.co/nvidia/Nemotron-3-Diarization) is NVIDIA's ~99M-parameter speaker diarizer (about 400 MB of weights). It answers **who spoke when** on a podcast, meeting, or call: up to **eight speakers**, overlapping speech, offline or streaming. Speaker channels are ordered by first arrival. It does **not** transcribe.

This notebook is the diarization counterpart of the existing [AIBackends Whisper notebook](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb). Whisper (and Granite Speech ASR) write words; this model writes speaker timelines and RTTM. Pair them after the fact for speaker-labeled transcripts — this notebook does not load an ASR model and does not depend on any other PR.

**Licence:** [OpenMDW-1.1](https://openmdw.ai/license/1-1/). Permissive use of the model materials, including commercial use. If you redistribute the weights or card artifacts, keep the license text and origin notices. Patent/copyright litigation against a provider terminates the grant. Generated outputs are unrestricted. The materials are provided as-is; you are responsible for clearing third-party rights in the audio you run.

**Runtime:** *Runtime > Change runtime type > T4 GPU* (CPU also works). The free T4 has 16 GB of VRAM and no native bfloat16. This notebook loads **float16** on T4 and **float32** on CPU. Weights are ≪1 GB; a short clip stays well under 4 GB. No FlashAttention 2.

**Links:** [model card](https://huggingface.co/nvidia/Nemotron-3-Diarization) · [blog](https://huggingface.co/blog/nvidia/nemotron-diarization) · [Transformers docs](https://huggingface.co/docs/transformers/en/model_doc/nemotron3_diarization) · [demo](https://huggingface.co/spaces/nvidia/nemotron-diarization) · [OpenMDW-1.1](https://openmdw.ai/license/1-1/)

In [ ]:
%pip install -q "transformers==5.20.1" librosa soundfile

# Native AutoModelForAudioFrameClassification / AutoProcessor support is in
# transformers>=5.18 (documented for 5.20.2). 5.20.1 matches the other T4 notebooks.
# The feature extractor imports librosa. soundfile reads the WAV that espeak-ng writes.

# espeak-ng synthesises a two-speaker meeting clip below (GPL, no extra Python deps).
!apt-get -qq install -y espeak-ng

> **Restart the runtime now:** *Runtime > Restart session*, then continue from the next cell.
> Colab has already imported its preinstalled transformers. The upgrade is invisible until the process restarts.

In [ ]:
import shutil
import subprocess
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from packaging.version import Version

assert Version(transformers.__version__) >= Version("5.18.0"), (
    f"transformers {transformers.__version__} is loaded; Nemotron-3 Diarization needs 5.18+. "
    "Run the install cell, then Runtime > Restart session."
)

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    DEVICE = "cuda"
    # T4 (sm_75) has no native bfloat16. The card's NeMo evals use bf16; float16 is the T4 path.
    DTYPE = torch.bfloat16 if major >= 8 else torch.float16
    print("GPU:", torch.cuda.get_device_name(), f"sm_{major}{minor}", "| dtype:", DTYPE)
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    DEVICE = "cpu"
    DTYPE = torch.float32
    print("No GPU — running on CPU in float32. A free T4 is faster; this ~99M model still fits RAM.")

TARGET_SR = 16_000
print("transformers", transformers.__version__)
print("torch", torch.__version__)
print("device:", DEVICE, "| dtype:", DTYPE)
print("espeak-ng:", shutil.which("espeak-ng") or shutil.which("espeak") or "not found")


def print_memory(prefix=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        peak = torch.cuda.max_memory_allocated() / 1024**3
        print(f"{prefix}VRAM allocated: {allocated:.2f} GB | peak: {peak:.2f} GB")
    else:
        import resource

        rss_kb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        print(f"{prefix}CPU RSS peak: {rss_kb / 1024 / 1024:.2f} GB")

## 1. Load Nemotron-3 Diarization

`AutoProcessor` + `AutoModelForAudioFrameClassification`, as on the [model card](https://huggingface.co/nvidia/Nemotron-3-Diarization) and the [Transformers docs](https://huggingface.co/docs/transformers/en/model_doc/nemotron3_diarization). The first run downloads about 400 MB. Offline mode chunks a whole recording in one `forward`; streaming is a later cell.

In [ ]:
from transformers import AutoModelForAudioFrameClassification, AutoProcessor

MODEL_ID = "nvidia/Nemotron-3-Diarization"

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForAudioFrameClassification.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
).to(DEVICE).eval()

sampling_rate = processor.feature_extractor.sampling_rate
assert sampling_rate == TARGET_SR

print("architecture:", model.config.architectures)
print("max speakers:", model.config.head_config.num_speakers)
print("dtype:", model.dtype, "| device:", model.device)
print_memory()

## 2. License-clean two-speaker meeting clip

The card's Transformers snippet points at `hf-internal-testing/dummy-audio-samples` `diarization_example.mp3`. That fixture is not in the dataset's published provenance table, so this notebook does **not** download it.

Instead we synthesise a short standup with `espeak-ng`: two distinct voices, a known 0.35 s overlap, and a third turn after a pause. Robotic TTS is enough to show speaker changes and overlap. Swap in your own 16 kHz mono recording for a real podcast or meeting.

In [ ]:
SAMPLE_DIR = Path("nemotron_diarization_samples")
SAMPLE_DIR.mkdir(exist_ok=True)

TURNS = [
    {
        "speaker": 0,
        "voice": "en-us+f3",
        "text": "Welcome to the weekly product standup. Let's start with the release notes.",
        "start": 0.00,
        "file": SAMPLE_DIR / "turn0.wav",
    },
    {
        "speaker": 1,
        "voice": "en-gb+m3",
        "text": "The beta went out on Tuesday. Two customers reported a login timeout.",
        "start": None,  # filled after we know turn 0 duration
        "file": SAMPLE_DIR / "turn1.wav",
    },
    {
        "speaker": 0,
        "voice": "en-us+f3",
        "text": "We should patch that before Friday. I will take the API side.",
        "start": None,
        "file": SAMPLE_DIR / "turn2.wav",
    },
]
OVERLAP_S = 0.35
GAP_S = 0.45


def synthesize(path, text, voice):
    exe = shutil.which("espeak-ng") or shutil.which("espeak")
    if not exe:
        raise RuntimeError("espeak-ng is missing; rerun the install cell.")
    subprocess.run([exe, "-v", voice, "-s", "140", "-w", str(path), text], check=True)
    return path


def load_mono16k(path):
    wav, sr = sf.read(path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != TARGET_SR:
        # Linear resample is enough for espeak; keeps the notebook off torchaudio.
        n = int(round(len(wav) * TARGET_SR / sr))
        wav = np.interp(np.linspace(0, len(wav) - 1, n), np.arange(len(wav)), wav).astype(
            np.float32
        )
    return wav


for turn in TURNS:
    synthesize(turn["file"], turn["text"], turn["voice"])
    turn["wav"] = load_mono16k(turn["file"])
    turn["duration"] = len(turn["wav"]) / TARGET_SR

TURNS[1]["start"] = max(TURNS[0]["duration"] - OVERLAP_S, 0.0)
TURNS[2]["start"] = TURNS[1]["start"] + TURNS[1]["duration"] + GAP_S

total_s = max(t["start"] + t["duration"] for t in TURNS)
mix = np.zeros(int(round(total_s * TARGET_SR)) + 1, dtype=np.float32)
expected = []
for turn in TURNS:
    start_i = int(round(turn["start"] * TARGET_SR))
    end_i = start_i + len(turn["wav"])
    if end_i > len(mix):
        mix = np.pad(mix, (0, end_i - len(mix)))
    mix[start_i:end_i] += turn["wav"]
    expected.append(
        {
            "Speaker": turn["speaker"],
            "Start": round(turn["start"], 2),
            "End": round(turn["start"] + turn["duration"], 2),
        }
    )

peak = np.max(np.abs(mix)) or 1.0
mix = (0.95 * mix / peak).astype(np.float32)
mix_path = SAMPLE_DIR / "standup_2spk.wav"
sf.write(mix_path, mix, TARGET_SR)

print(f"wrote {mix_path}  {len(mix) / TARGET_SR:.2f}s  {TARGET_SR} Hz mono")
print("ground-truth turns (synthesis offsets):")
for seg in expected:
    print(f"  speaker_{seg['Speaker']}: {seg['Start']:.2f}s - {seg['End']:.2f}s")
display(Audio(mix, rate=TARGET_SR))

## 3. Offline diarization → segments, RTTM, timeline

Card recipe: `processor(audio)` → `model(**inputs).logits` (one frame every 10 ms, 8 speaker channels) → `processor.extract_speaker_dict`. Speakers are numbered by first arrival. Overlap produces overlapping segments. A short same-speaker merge closes the gaps that robotic TTS (and real breaths) leave inside a turn.

In [ ]:
audio = mix  # 16 kHz mono float32 from the cell above; no extra decode step
inputs = processor(audio, sampling_rate=sampling_rate).to(model.device, dtype=model.dtype)

if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()

with torch.inference_mode():
    logits = model(**inputs).logits  # (1, num_frames, 8)

raw_segments = processor.extract_speaker_dict(logits, inputs.attention_mask)[0]


def merge_segments(segments, max_gap=0.50, min_dur=0.15):
    """Drop tiny blips, then join same-speaker runs separated by a short gap."""
    kept = [s for s in segments if s["End"] - s["Start"] >= min_dur]
    merged = []
    for seg in sorted(kept, key=lambda s: (s["Speaker"], s["Start"])):
        if (
            merged
            and merged[-1]["Speaker"] == seg["Speaker"]
            and seg["Start"] - merged[-1]["End"] <= max_gap
        ):
            merged[-1] = {**merged[-1], "End": seg["End"]}
        else:
            merged.append(dict(seg))
    merged.sort(key=lambda s: (s["Start"], s["Speaker"]))
    return merged


segments = merge_segments(raw_segments)
n_speakers = len({seg["Speaker"] for seg in segments})
print(
    f"logits {tuple(logits.shape)}  |  raw {len(raw_segments)} segments → "
    f"{len(segments)} after merge  |  {n_speakers} active speakers"
)
print_memory("after diarize  ")

print("\npredicted segments (merged):")
for seg in segments:
    print(f"  speaker_{seg['Speaker']}: {seg['Start']:.2f}s - {seg['End']:.2f}s")

In [ ]:
import matplotlib.pyplot as plt


def to_rttm(segments, file_id="standup_2spk"):
    lines = []
    for seg in segments:
        start = float(seg["Start"])
        dur = float(seg["End"]) - start
        if dur <= 0:
            continue
        lines.append(
            f"SPEAKER {file_id} 1 {start:.3f} {dur:.3f} <NA> <NA> speaker_{seg['Speaker']} <NA> <NA>"
        )
    return "\n".join(lines)


rttm = to_rttm(segments)
rttm_path = SAMPLE_DIR / "standup_2spk.rttm"
rttm_path.write_text(rttm + "\n")
print(f"NIST RTTM ({rttm_path}):")
print(rttm)

speakers = sorted({seg["Speaker"] for seg in segments} | {seg["Speaker"] for seg in expected})
colors = plt.cm.tab10.colors
fig, axes = plt.subplots(2, 1, figsize=(10, 3.6), sharex=True)

for ax, title, segs in (
    (axes[0], "ground truth (synthesis offsets)", expected),
    (axes[1], "Nemotron-3 Diarization", segments),
):
    for spk in speakers:
        spans = [
            (s["Start"], s["End"] - s["Start"]) for s in segs if s["Speaker"] == spk
        ]
        ax.broken_barh(spans, (spk - 0.35, 0.7), facecolor=colors[spk % 10], alpha=0.85)
    ax.set_yticks(speakers)
    ax.set_yticklabels([f"speaker_{s}" for s in speakers])
    ax.set_ylabel(title)
    ax.grid(axis="x", linestyle=":", alpha=0.4)

axes[1].set_xlabel("time (s)")
axes[0].set_title("speaker timeline")
fig.tight_layout()
plt.show()

## 4. Streaming pass on the same clip (optional)

Same checkpoint, `processor.set_streaming_mode("low_latency")` (1.04 s input buffer). Each `forward` takes one chunk and returns a `speaker_cache` for the next call. This is the live-meeting path; offline mode above is the podcast / recorded-standup path.

In [ ]:
processor.set_streaming_mode("low_latency")
print(f"streaming mode: {processor.streaming_mode}  |  latency: {processor.streaming_latency_ms} ms")


def streaming_chunks(wav):
    yield processor(
        wav[: processor.num_samples_first_audio_chunk],
        sampling_rate=sampling_rate,
        is_streaming=True,
        is_first_audio_chunk=True,
    )
    mel_frame_idx = processor.num_mel_frames_per_step
    start_idx = processor.audio_chunk_start(mel_frame_idx)
    while (end_idx := start_idx + processor.num_samples_per_audio_chunk) <= wav.shape[0]:
        yield processor(
            wav[start_idx:end_idx],
            sampling_rate=sampling_rate,
            is_streaming=True,
            is_first_audio_chunk=False,
        )
        mel_frame_idx += processor.num_mel_frames_per_step
        start_idx = processor.audio_chunk_start(mel_frame_idx)
    yield processor(
        wav[start_idx:],
        sampling_rate=sampling_rate,
        is_streaming=True,
        is_first_audio_chunk=False,
        is_last_audio_chunk=True,
    )


speaker_cache, stream_logits = None, []
with torch.inference_mode():
    for chunk_inputs in streaming_chunks(audio):
        chunk_inputs = chunk_inputs.to(model.device, dtype=model.dtype)
        outputs = model(**chunk_inputs, speaker_cache=speaker_cache)
        stream_logits.append(outputs.logits)
        speaker_cache = outputs.speaker_cache

stream_logits = torch.cat(stream_logits, dim=1)
stream_segments = processor.extract_speaker_dict(stream_logits)[0]
print(f"streaming logits {tuple(stream_logits.shape)}  |  {len(stream_segments)} segments")
for seg in stream_segments:
    print(f"  speaker_{seg['Speaker']}: {seg['Start']:.2f}s - {seg['End']:.2f}s")
print_memory("after streaming  ")

## 5. Caveats

**OpenMDW-1.1.** This is not Apache 2.0 and not MIT. Use of the model materials is governed by the [OpenMDW License Agreement v1.1](https://openmdw.ai/license/1-1/). Redistribution of the weights or card artifacts must keep that agreement and the origin notices. Outputs you generate are not restricted by the license. The grant terminates if you sue a provider for patent or copyright infringement over the materials (unless you are responding to a suit first brought against you). No warranty; you clear third-party rights in the audio you process.

**Eight speakers, not transcription.** The head has eight channels. A ninth speaker is not a new label — it will be merged or dropped. The model does not emit text. For speaker-labeled transcripts, run any ASR on the same clip (or on each diarized segment) and attach `speaker_N`.

**Pairing with Granite Speech ASR.** An [open PR](https://github.com/donvito/notebooks/pull/13) adds `colab/granite-speech-5.0-2b-asr-translation.ipynb` for keyword-biased ASR and speech translation with [`ibm-granite/granite-speech-5.0-2b`](https://huggingface.co/ibm-granite/granite-speech-5.0-2b) (Apache 2.0). Diarize here, transcribe there, join on time. This notebook does not import that file and does not require that PR to merge.

**Not the Whisper notebook.** [`AIBackends-Whisper-audio-transcription.ipynb`](https://github.com/donvito/notebooks/blob/main/colab/AIBackends-Whisper-audio-transcription.ipynb) is local faster-whisper ASR plus PII redaction. It has no speaker IDs. This notebook has speaker IDs and no words. Use Whisper (or Granite Speech) when you need text; use Nemotron-3 Diarization when you need who-spoke-when.

**Synthetic clip.** `espeak-ng` is robotic and the ground-truth bars are mix offsets, not a scored DER. Swap in a real 16 kHz mono meeting or podcast for a fairer check. The card's `diarization_example.mp3` is the Transformers doc fixture if you want that path.

**T4 / CPU.** Float16 on T4, float32 on CPU. No FlashAttention 2. Peak VRAM after a short clip should stay well under 16 GB (weights ≪1 GB). Chunked inference has no fixed maximum duration; keep Colab sessions to minutes, not hours.